# Dataset Pipeline

In [1]:
import torch
import numpy as np
import evaluate

from datasets import load_dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)


# Check device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Y:\Python\AI ML\Month 2\week 7-Deep Learning\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU: NVIDIA GeForce GTX 1650


In [2]:
dataset = load_dataset("stanfordnlp/imdb")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [3]:
# Take only 5,000 examples from the original training set
train_5000 = dataset["train"].shuffle(seed=42).select(range(5000))

# Stratified 80/20 split
split_dataset = train_5000.train_test_split(
    test_size=0.2,
    seed=42,
    stratify_by_column="label"
)

dataset = DatasetDict({
    "train": split_dataset["train"],
    "validation": split_dataset["test"]
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 4000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1000
    })
})


In [4]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

In [5]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print(tokenized_dataset)

Map: 100%|██████████| 1000/1000 [00:00<00:00, 4030.29 examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 4000
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1000
    })
})


In [6]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

# Evaluvate Without fine Tuning

In [18]:
model_without_fine_tuning = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1283.43it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
accuracy = evaluate.load("accuracy")

In [9]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    return accuracy.compute(
        predictions=predictions,
        references=labels
    )

In [19]:
Evaluvate_Without_Fine_Tuning = Trainer(
    model=model_without_fine_tuning,

    train_dataset=tokenized_dataset["train"],

    eval_dataset=tokenized_dataset["validation"],

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

print(Evaluvate_Without_Fine_Tuning.evaluate())

Training Loss,Validation Loss,Step,Accuracy
No log,0.694524,0,0.480000


{'eval_loss': 0.6945239305496216, 'eval_accuracy': 0.48}


# Training and Fine Tuning

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 7801.61it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
training_args = TrainingArguments(
    output_dir="./distilbert-imdb",

    learning_rate=2e-5,
    weight_decay=0.01,

    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,

    num_train_epochs=2,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="accuracy",
    greater_is_better=True,

    logging_strategy="epoch",

    save_total_limit=2,

    report_to="none",

    fp16=True
)

In [11]:
trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=tokenized_dataset["train"],

    eval_dataset=tokenized_dataset["validation"],

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

In [12]:
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.402687,0.328052,0.855000
2,0.235124,0.325056,0.862000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.85it/s]


In [13]:
eval_results = trainer.evaluate()

print(eval_results)

Training Loss,Validation Loss,Epoch,Accuracy
0.235124,0.325056,2,0.862000


{'eval_loss': 0.3250555396080017, 'eval_accuracy': 0.862}


In [14]:
load_best_model_at_end=True

In [15]:
print("Best checkpoint:")
print(trainer.state.best_model_checkpoint)

print("\nBest metric:")
print(trainer.state.best_metric)

Best checkpoint:
./distilbert-imdb\checkpoint-250

Best metric:
0.862


In [16]:
text = "This movie was absolutely fantastic. I loved it!"

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=256
)

# Put inputs on the same device as the model
inputs = {
    key: value.to(trainer.model.device)
    for key, value in inputs.items()
}

trainer.model.eval()

with torch.no_grad():
    outputs = trainer.model(**inputs)

logits = outputs.logits

prediction = torch.argmax(
    logits,
    dim=-1
).item()

id2label = {
    0: "NEGATIVE",
    1: "POSITIVE"
}

print("Logits:", logits)
print("Prediction:", id2label[prediction])

Logits: tensor([[-1.6338,  1.6006]], device='cuda:0')
Prediction: POSITIVE
